# Calibration analysis

This notebook is the centerpiece of the v1 writeup. It walks through:

1. Loading the eval results from the train split
2. Plotting the raw (uncalibrated) reliability diagram
3. Fitting Platt scaling
4. Plotting the calibrated reliability diagram
5. Showing ECE / Brier improvements
6. Validating on the dev split
7. Deriving the threshold policy from a cost model

Run this after `python -m src.eval.harness` has produced `eval_results/train.jsonl` and `eval_results/dev.jsonl`.

In [ ]:
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

from src.eval.calibration import (
    expected_calibration_error, fit_platt, load_eval_results, reliability_diagram_data
)

## Load eval results

In [ ]:
train_raw, train_correct = load_eval_results(Path('../eval_results/train.jsonl'))
dev_raw, dev_correct = load_eval_results(Path('../eval_results/dev.jsonl'))
print(f'Train: {len(train_raw)} records, {train_correct.mean():.1%} correct')
print(f'Dev:   {len(dev_raw)} records, {dev_correct.mean():.1%} correct')

## Reliability diagram: before vs after

Each bar is a confidence bin. Bar height = empirical accuracy in that bin. Perfectly calibrated = diagonal line.

In [ ]:
def plot_reliability(per_bin, ax, title):
    centers = [(b['lo'] + b['hi']) / 2 for b in per_bin if b['n'] > 0]
    accs = [b['accuracy'] for b in per_bin if b['n'] > 0]
    confs = [b['mean_confidence'] for b in per_bin if b['n'] > 0]
    ns = [b['n'] for b in per_bin if b['n'] > 0]

    ax.plot([0, 1], [0, 1], 'k--', alpha=0.4, label='perfect')
    ax.bar(centers, accs, width=0.08, alpha=0.6, edgecolor='black', label='accuracy')
    ax.scatter(confs, accs, s=[n*5 for n in ns], color='red', zorder=3, label='bin mean')
    ax.set_xlabel('Predicted confidence')
    ax.set_ylabel('Empirical accuracy')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_title(title)
    ax.legend(loc='upper left')
    ax.grid(alpha=0.3)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

raw_per_bin = reliability_diagram_data(train_raw / 100, train_correct)
raw_ece, _ = expected_calibration_error(train_raw / 100, train_correct)
plot_reliability(raw_per_bin, axes[0], f'Uncalibrated (ECE = {raw_ece:.3f})')

model = fit_platt(train_raw, train_correct)
cal_train = model.predict_proba(train_raw)
cal_per_bin = reliability_diagram_data(cal_train, train_correct)
plot_reliability(cal_per_bin, axes[1], f'Calibrated (ECE = {model.train_ece:.3f})')

plt.tight_layout()
plt.savefig('../eval_results/reliability_train.png', dpi=150)
plt.show()

## Threshold policy

Once calibration is in place, map calibrated probability to action. See EVALUATION.md for the cost model.